In [ ]:
code = """
import json
import boto3
import os
import onnxruntime as ort
import numpy as np
from transformers import AutoTokenizer

S3_BUCKET = os.environ.get('BUCKET_NAME')
LOCAL_PATH = '/tmp/model'

s3 = boto3.client('s3')
tokenizer = None
ort_session = None

def download_model():
    if not os.path.exists(LOCAL_PATH):
        os.makedirs(LOCAL_PATH)
    
    files = ['model_quantized.onnx', 'config.json', 'tokenizer.json', 
             'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model']
    
    print("Downloading model from S3...")
    for f in files:
        dest = os.path.join(LOCAL_PATH, f)
        if not os.path.exists(dest):
            try:
                s3.download_file(S3_BUCKET, f, dest)
            except:
                print(f"File {f} not found in S3 bucket {S3_BUCKET}")

def load_model():
    global tokenizer, ort_session
    download_model()
    
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH)
    
    if ort_session is None:
        model_path = os.path.join(LOCAL_PATH, "model_quantized.onnx")
        ort_session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def handler(event, context):
    output_payload = {
            "user_id": user_id,
            "sender": sender,
            "message": message,
            "veredict": Veredict.UNKNOWN.value,
            "reason": "Not processed",
        }

    try:
        if 'body' in event:
            if isinstance(event['body'], str):
                input_data = json.loads(event['body'])
            else:
                input_data = event['body']
        else:
            input_data = event
        
        response_data.update(input_data)
        
        text = response_data.get('message', '')
        
        if not text or not isinstance(text, str):
            output_payload['veredict'] = "UNKNOWN"
            output_payload['reason'] = "No text provided in 'message' field"
            return output_payload

        load_model()
        
        inputs = tokenizer(text, return_tensors="np", padding=True, truncation=True)
        onnx_inputs = {k: v.astype(np.int64) for k, v in inputs.items()}
        
        logits = ort_session.run(None, onnx_inputs)[0][0]
        probs = softmax(logits)
        
        spam_prob = float(probs[1])
        ham_prob = float(probs[0])
        
        if spam_prob > 0.5:
            output_payload['veredict'] = "SPAM"
            output_payload['reason'] = "AI detected malicious content pattern"
        else:
            output_payload['veredict'] = "HAM"
            output_payload['reason'] = "Content appears safe"
            
        output_payload['details'] = f"Confidence: {spam_prob*100:.2f}% (Spam Score)"

        return output_payload

    except Exception as e:
        print(f"FATAL ERROR: {e}")
        raise e
"""

with open("lambda_package/lambda_function.py", "w") as f:
    f.write(code)

!cd lambda_package && zip -r9 ../deployment_package.zip .
print("ZIP updated. Download it from the menu: deployment_package.zip")

In [ ]:
import os
import shutil
from enum import Enum

!rm -rf lambda_package deployment_package.zip
!mkdir -p lambda_package

print("Instalando dependencias optimizadas...")

# 1. NO instalamos boto3 (AWS ya lo tiene).
# 2. Instalamos onnxruntime y numpy.
!pip install --target ./lambda_package onnxruntime numpy

# 3. Instalamos transformers SIN dependencias pesadas y bajamos solo lo vital
!pip install --target ./lambda_package transformers --no-deps
!pip install --target ./lambda_package tokenizers filelock huggingface-hub safetensors pyyaml regex packaging requests charset-normalizer idna urllib3 certifi

print("Limpiando archivos basura para reducir tamaño...")

def clean_package(path):
    # Lista negra de carpetas que ocupan espacio y no sirven en Lambda
    useless_dirs = ['__pycache__', 'tests', 'test', 'docs', 'examples', 
                    'botocore', 'boto3', 's3transfer', 'nvidia', 'bin', 'scipy']
    
    for root, dirs, files in os.walk(path, topdown=False):
        for name in list(dirs):
            if name in useless_dirs or name.endswith('.dist-info') or name.endswith('.egg-info'):
                shutil.rmtree(os.path.join(root, name))
        for name in files:
            if name.endswith('.pyc') or name.endswith('.pyo'):
                os.remove(os.path.join(root, name))

clean_package('./lambda_package')

code = """
import json
import boto3
import os
import onnxruntime as ort
import numpy as np
from transformers import AutoTokenizer
from common.notification import Veredict

S3_BUCKET = os.environ.get('BUCKET_NAME')
LOCAL_PATH = '/tmp/model'

s3 = boto3.client('s3')
tokenizer = None
ort_session = None

def download_model():
    if not os.path.exists(LOCAL_PATH):
        os.makedirs(LOCAL_PATH)
    
    files = ['model_quantized.onnx', 'config.json', 'tokenizer.json', 
             'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model']
    
    print("Downloading model from S3...")
    for f in files:
        dest = os.path.join(LOCAL_PATH, f)
        if not os.path.exists(dest):
            try:
                s3.download_file(S3_BUCKET, f, dest)
            except:
                print(f"File {f} not found in S3 bucket {S3_BUCKET}")

def load_model():
    global tokenizer, ort_session
    download_model()
    
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH)
    
    if ort_session is None:
        model_path = os.path.join(LOCAL_PATH, "model_quantized.onnx")
        ort_session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def handler(event, context):

    try:
        if 'body' in event:
            if isinstance(event['body'], str):
                body = json.loads(event['body'])
            else:
                body = event['body']
        else:
            body = event

        output_payload = {
            "user_id": body.get("user_id", ""),
            "sender": body.get("sender", ""),
            "message": body.get("message", ""),
            "veredict": Veredict.UNKNOWN.value,
            "reason": "Not processed",
        }
        
        output_payload.update(body)
        
        text = output_payload.get('message', '')
        
        if not text or not isinstance(text, str):
            output_payload['reason'] = "No text provided in 'message' field"
            return output_payload

        load_model()
        
        inputs = tokenizer(text, return_tensors="np", padding=True, truncation=True)
        onnx_inputs = {k: v.astype(np.int64) for k, v in inputs.items()}
        
        logits = ort_session.run(None, onnx_inputs)[0][0]
        probs = softmax(logits)
        
        spam_prob = float(probs[1])
        ham_prob = float(probs[0])
        
        if spam_prob > 0.8:
            output_payload['veredict'] = Veredict.MALICIOUS.value
            output_payload['reason'] = "AI detected malicious content pattern"
        elif spam_prob > 0.5:
            output_payload['veredict'] = Veredict.SUSPICIOUS.value
            output_payload['reason'] = "Content may be malicious"
        else:
            output_payload['veredict'] = Veredict.SAFE.value
            output_payload['reason'] = "Content appears safe"
            
        output_payload['details'] = f"Confidence: {spam_prob*100:.2f}% (Spam Score)"

        return output_payload

    except Exception as e:
        print(f"FATAL ERROR: {e}")
        raise e
"""

with open("lambda_package/lambda_function.py", "w") as f:
    f.write(code)

!cd lambda_package && zip -r9 ../deployment_package.zip .
print("ZIP updated. Download it from the menu: deployment_package.zip")

In [ ]:
code = """
import json
import boto3
import os
import onnxruntime as ort
import numpy as np
from transformers import AutoTokenizer
from common.notification import Veredict

S3_BUCKET = os.environ.get('BUCKET_NAME')
LOCAL_PATH = '/tmp/model'

s3 = boto3.client('s3')
tokenizer = None
ort_session = None

def download_model():
    if not os.path.exists(LOCAL_PATH):
        os.makedirs(LOCAL_PATH)
    
    files = ['model_quantized.onnx', 'config.json', 'tokenizer.json', 
             'tokenizer_config.json', 'special_tokens_map.json', 'sentencepiece.bpe.model']
    
    print("Downloading model from S3...")
    for f in files:
        dest = os.path.join(LOCAL_PATH, f)
        if not os.path.exists(dest):
            try:
                s3.download_file(S3_BUCKET, f, dest)
            except:
                print(f"File {f} not found in S3 bucket {S3_BUCKET}")

def load_model():
    global tokenizer, ort_session
    download_model()
    
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH)
    
    if ort_session is None:
        model_path = os.path.join(LOCAL_PATH, "model_quantized.onnx")
        ort_session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def handler(event, context):

    try:
        if 'body' in event:
            if isinstance(event['body'], str):
                body = json.loads(event['body'])
            else:
                body = event['body']
        else:
            body = event

        output_payload = {
            "user_id": body.get("user_id", ""),
            "sender": body.get("sender", ""),
            "message": body.get("message", ""),
            "veredict": Veredict.UNKNOWN.value,
            "reason": "Not processed",
        }
        
        output_payload.update(body)
        
        text = output_payload.get('message', '')
        
        if not text or not isinstance(text, str):
            output_payload['reason'] = "No text provided in 'message' field"
            return output_payload

        load_model()
        
        inputs = tokenizer(text, return_tensors="np", padding=True, truncation=True)
        onnx_inputs = {k: v.astype(np.int64) for k, v in inputs.items()}
        
        logits = ort_session.run(None, onnx_inputs)[0][0]
        probs = softmax(logits)
        
        spam_prob = float(probs[1])
        ham_prob = float(probs[0])
        
        if spam_prob > 0.8:
            output_payload['veredict'] = Veredict.MALICIOUS.value
            output_payload['reason'] = "AI detected malicious content pattern"
        elif spam_prob > 0.5:
            output_payload['veredict'] = Veredict.SUSPICIOUS.value
            output_payload['reason'] = "Content may be malicious"
        else:
            output_payload['veredict'] = Veredict.SAFE.value
            output_payload['reason'] = "Content appears safe"
            
        output_payload['details'] = f"Confidence: {spam_prob*100:.2f}% (Spam Score)"

        return output_payload

    except Exception as e:
        print(f"FATAL ERROR: {e}")
        raise e
"""

with open("lambda_package/lambda_function.py", "w") as f:
    f.write(code)

!cd lambda_package && zip -r9 ../deployment_package.zip .
print("ZIP updated. Download it from the menu: deployment_package.zip")